In [ ]:
import os
import pickle
import mediapipe as mp
import cv2
import matplotlib.pyplot as plt

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    min_detection_confidence=0.1,
    max_num_hands=2
)

DATA_DIR = './data'

data = []
labels = []

for dir_ in os.listdir(DATA_DIR):

    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):

        data_aux = []
        x_ = []
        y_ = []

        img = cv2.imread(os.path.join(DATA_DIR, dir_, img_path))

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)
        # Only accept images containing exactly two hands
        if results.multi_hand_landmarks and len(results.multi_hand_landmarks) == 2:

            # Extract coordinates from both hands
            for hand_landmarks in results.multi_hand_landmarks:
                for landmark in hand_landmarks.landmark:
                    x_.append(landmark.x)
                    y_.append(landmark.y)

            # Normalize all landmarks using the same minimum values
            for hand_landmarks in results.multi_hand_landmarks:
                for landmark in hand_landmarks.landmark:

                    x = landmark.x
                    y = landmark.y

                    data_aux.append(x - min(x_))
                    data_aux.append(y - min(y_))

            # Make sure every sample has 84 features
            if len(data_aux) == 84:
                data.append(data_aux)
                labels.append(dir_)
            else:
                print("Invalid feature count:", len(data_aux))

        else:
            print("Skipping image:", img_path, "- Two hands not detected")

# Save dataset
f = open('data.pickle', 'wb')
pickle.dump({'data': data, 'labels': labels}, f)
f.close()

hands.close()

print("Total samples:", len(data))
print("Feature count:", len(data[0]) if data else 0)